# JUNE 25th-anniversary analysis — Step 3: count citations

**What this does:** for every JUNE item, counts how many later papers cite it,
using OpenAlex (a free, open index of scholarly papers). For each item we report:
- `citations_total` — all citations so far
- `citations_first_5_years` — citations from papers published in the item's
  publication year or the 4 years after (e.g., 2010–2014 for a 2010 item).
  Left **blank** for items too new to have a complete 5-year window.
- `citations_by_year` — the year-by-year counts, e.g. "2011: 2; 2012: 5"

As a cross-check we also record NIH iCite's count, which only counts
citations from papers indexed in PubMed (so it's usually lower).

**What you need to do:** run each cell from top to bottom (a few minutes).

**What goes in:** `data/june_matched.csv` (from Step 2)

**What comes out:** `data/june_citations.csv` — one row per item

Citation counts grow over time, so the file records the date they were
retrieved (`citations_retrieved`). Report that date in the manuscript.

## Settings — the only cell you should normally need to edit

In [1]:
MATCHED_CSV = "data/june_matched.csv"
OUTPUT_CSV = "data/june_citations.csv"

# The "first 5 years" window: the publication year plus this many years after.
YEARS_AFTER_PUBLICATION = 4

# OpenAlex allows about 1,000 requests a day without an API key, which is
# enough for one run (this notebook uses roughly 100). For more, get a free
# key at https://openalex.org/settings/api and paste it here.
OPENALEX_API_KEY = ""
CONTACT_EMAIL = ""   # optional; OpenAlex and iCite like to know who's asking

CACHE_DIR = "cache"

## Setup (no need to edit below this line)

In [2]:
import datetime
import hashlib
import json
import os
import time

import pandas as pd
import requests

TODAY = datetime.date.today()


def get_json(url, params):
    """Download JSON, keeping a copy in cache/ for the rest of today.

    Re-running the notebook on the same day re-uses today's downloads (so it's
    fast and doesn't use up the daily OpenAlex allowance); the next day it
    downloads fresh citation counts."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    key = hashlib.md5((url + json.dumps(params, sort_keys=True)).encode()).hexdigest()
    path = os.path.join(CACHE_DIR, f"api-{TODAY}-{key}.json")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    full = dict(params)
    if "openalex" in url and OPENALEX_API_KEY:
        full["api_key"] = OPENALEX_API_KEY
    if CONTACT_EMAIL:
        full["mailto"] = CONTACT_EMAIL
    for attempt in range(1, 5):
        resp = requests.get(url, params=full, timeout=60)
        if resp.status_code not in (429, 500, 502, 503, 504) or attempt == 4:
            break
        print(f"  server busy ({resp.status_code}); waiting {10 * attempt} seconds…")
        time.sleep(10 * attempt)
    if resp.status_code == 429:
        raise RuntimeError("OpenAlex's daily limit is used up. Try again tomorrow, "
                           "or add a free API key in the Settings cell.")
    resp.raise_for_status()
    data = resp.json()
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    return data


def batches(items, size):
    items = list(items)
    for start in range(0, len(items), size):
        yield items[start:start + size]

## Step A: find each JUNE item in OpenAlex

We look items up by PubMed ID and by DOI. A few newer articles have **two**
OpenAlex records (one from PubMed, one from the DOI), each holding some of
the citations. We keep both and count each citing paper once.

In [3]:
def short_id(url):
    """'https://openalex.org/W2128426344' → 'W2128426344'"""
    return url.rsplit("/", 1)[-1]


def find_openalex_works(df):
    works_for_item = {item: set() for item in df["item_id"]}

    by_pmid = {p: item for p, item in zip(df["pmid"], df["item_id"]) if p}
    for batch in batches(by_pmid, 50):
        data = get_json("https://api.openalex.org/works",
                        {"filter": "ids.pmid:" + "|".join(batch), "per_page": 100, "select": "id,ids"})
        for w in data["results"]:
            pmid = short_id(w["ids"].get("pmid", ""))
            if pmid in by_pmid:
                works_for_item[by_pmid[pmid]].add(short_id(w["id"]))

    by_doi = {d.lower(): item for d, item in zip(df["doi"], df["item_id"]) if d}
    for batch in batches(by_doi, 50):
        data = get_json("https://api.openalex.org/works",
                        {"filter": "doi:" + "|".join(batch), "per_page": 100, "select": "id,doi"})
        for w in data["results"]:
            doi = (w.get("doi") or "").lower().replace("https://doi.org/", "")
            if doi in by_doi:
                works_for_item[by_doi[doi]].add(short_id(w["id"]))

    return works_for_item

## Step B: download the papers that cite JUNE

Rather than asking about one JUNE item at a time (~700 requests), we ask for
every paper that cites any of 50 JUNE items at once, along with its
publication year and reference list. The reference list tells us which
JUNE items each citing paper cites.

In [4]:
def download_citing_papers(all_june_works):
    citing = {}   # citing paper ID → (year, set of JUNE works it cites)
    batch_list = list(batches(sorted(all_june_works), 50))
    for n, batch in enumerate(batch_list, 1):
        cursor, pages = "*", 0
        while cursor:
            data = get_json("https://api.openalex.org/works",
                            {"filter": "cites:" + "|".join(batch), "per_page": 200, "cursor": cursor,
                             "select": "id,publication_year,referenced_works"})
            for w in data["results"]:
                cited = {short_id(r) for r in w.get("referenced_works") or []} & set(batch)
                year, prev = w.get("publication_year"), citing.get(short_id(w["id"]), (None, set()))
                citing[short_id(w["id"])] = (year or prev[0], prev[1] | cited)
            cursor = data["meta"].get("next_cursor") if data["results"] else None
            pages += 1
        print(f"  batch {n}/{len(batch_list)}: {pages} page(s)")
    return citing

## Step C: count citations per item

For each JUNE item, we collect the citing papers of all its OpenAlex records
(counting each citing paper once), then count them by year.

In [5]:
def count_citations(df, works_for_item, citing):
    cited_by = {}   # JUNE work → set of citing paper IDs
    for paper, (year, june_works) in citing.items():
        for w in june_works:
            cited_by.setdefault(w, set()).add(paper)

    last_complete_year = TODAY.year - 1   # this year isn't over yet
    rows = []
    for r in df.itertuples():
        works = works_for_item[r.item_id]
        papers = set().union(*(cited_by.get(w, set()) for w in works)) if works else set()
        per_year = pd.Series([citing[p][0] for p in papers], dtype="float").value_counts().sort_index()
        pub_year = int(r.year_label)
        window_end = pub_year + YEARS_AFTER_PUBLICATION
        rows.append({
            "item_id": r.item_id,
            "openalex_ids": "; ".join(sorted(works)),
            "citations_total": len(papers) if works else None,
            "citations_first_5_years": (int(per_year[(per_year.index >= pub_year) & (per_year.index <= window_end)].sum())
                                        if works and window_end <= last_complete_year else None),
            "citations_by_year": "; ".join(f"{int(y)}: {c}" for y, c in per_year.items()),
        })
    return pd.DataFrame(rows)

## Step D: cross-check with NIH iCite

iCite only counts citations from papers in PubMed, so its numbers are
usually lower than OpenAlex's. Big differences in the *other* direction
(iCite much higher) would suggest OpenAlex is missing something.

In [6]:
def icite_counts(pmids):
    counts = {}
    for batch in batches([p for p in pmids if p], 200):
        data = get_json("https://icite.od.nih.gov/api/pubs", {"pmids": ",".join(batch), "fl": "pmid,citation_count"})
        for rec in data.get("data", []):
            counts[str(rec["pmid"])] = rec.get("citation_count")
    return counts

## Run everything

In [7]:
def build_citations():
    df = pd.read_csv(MATCHED_CSV, keep_default_na=False, dtype=str, encoding="utf-8-sig")
    print(f"Read {len(df)} items.")

    works_for_item = find_openalex_works(df)
    found = sum(bool(w) for w in works_for_item.values())
    print(f"Found {found} of {len(df)} items in OpenAlex.")

    all_works = set().union(*works_for_item.values())
    print(f"Downloading papers that cite {len(all_works)} OpenAlex records…")
    citing = download_citing_papers(all_works)
    print(f"{len(citing)} citing papers.")

    counts = count_citations(df, works_for_item, citing)
    icite = icite_counts(df["pmid"])

    keep = ["item_id", "year_label", "volume", "issue", "item_type", "is_fun_workshop",
            "title", "first_author", "pmid", "doi"]
    out = df[keep].merge(counts, on="item_id")
    out["icite_citations"] = out["pmid"].map(icite)
    out["citations_retrieved"] = str(TODAY)
    for col in ["citations_total", "citations_first_5_years", "icite_citations"]:
        out[col] = pd.to_numeric(out[col]).astype("Int64")

    out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\nDone → {OUTPUT_CSV}")
    print(f"Items not found in OpenAlex (no citation count): {out['citations_total'].isna().sum()}")
    return out


citations = build_citations()
citations.head(10)

Read 716 items.
Found 668 of 716 items in OpenAlex.
  batch 1/14: 3 page(s)
  batch 2/14: 2 page(s)
  batch 3/14: 3 page(s)
  batch 4/14: 2 page(s)
  batch 5/14: 2 page(s)
  batch 6/14: 5 page(s)
  batch 7/14: 2 page(s)
  batch 8/14: 2 page(s)
  batch 9/14: 2 page(s)
  batch 10/14: 2 page(s)
  batch 11/14: 1 page(s)
  batch 12/14: 1 page(s)
  batch 13/14: 1 page(s)
  batch 14/14: 1 page(s)
3302 citing papers.

Done → data/june_citations.csv
Items not found in OpenAlex (no citation count): 48


,item_id,year_label,volume,issue,item_type,is_fun_workshop,title,first_author,pmid,doi,openalex_ids,citations_total,citations_first_5_years,citations_by_year,icite_citations,citations_retrieved
0,JUNE-01-1-01,2002,1,1,editorial,False,Introducing The Journal of Undergraduate Neuro...,Lom,23495292,,W183389940,9,0,2009: 1; 2012: 2; 2013: 1; 2015: 1; 2018: 1; 2...,9,2026-09-27
1,JUNE-01-1-02,2002,1,1,article,False,The Emergence of Non-Match-to-Sample Behavior ...,Gittis,23493448,,W2123891249,2,0,2018: 1; 2024: 1,2,2026-09-27
2,JUNE-01-1-03,2002,1,1,article,False,And The Winner Is: Inviting Hollywood into the...,Wiertelak,23493171,,W1719925799,12,1,2005: 1; 2010: 1; 2011: 1; 2015: 3; 2016: 1; 2...,7,2026-09-27
3,JUNE-01-1-04,2002,1,1,article,False,Exploring Neurogenesis in Crustaceans,Paul,23493254,,W2186889906,3,1,2003: 1; 2012: 1; 2024: 1,2,2026-09-27
4,JUNE-01-1-05,2002,1,1,article,False,Action Potentials in a Giant Algal Cell: A Com...,Johnson,23493389,,W2131146534,30,2,2004: 1; 2006: 1; 2007: 4; 2008: 1; 2012: 3; 2...,19,2026-09-27
5,JUNE-01-1-06,2002,1,1,review,False,Book Review: Mood Genes,Bernd,,,,<NA>,<NA>,,<NA>,2026-09-27
6,JUNE-01-1-07,2002,1,1,review,False,Textbook Review: Biological Psychology,Nolan,,,,<NA>,<NA>,,<NA>,2026-09-27
7,JUNE-01-1-08,2002,1,1,review,False,Media Review: Crawdad: A CD-ROM Lab Manual for...,Parfitt,,,,<NA>,<NA>,,<NA>,2026-09-27
8,JUNE-01-1-09,2002,1,1,review,False,Media Review: The Secret Life of the Brain,Brosnan-Watters,,,,<NA>,<NA>,,<NA>,2026-09-27
9,JUNE-01-2-01,2003,1,2,editorial,False,Introductory Neuroscience – Courses in an Evol...,Wiertelak,23741202,,W141322187,5,0,2008: 1; 2009: 2; 2011: 1; 2017: 1,6,2026-09-27


## Check the results

Things to look for:
- items not found in OpenAlex: mostly media reviews and editorials that
  aren't in PubMed and have no DOI
- OpenAlex and iCite should broadly agree on which items are cited most
- a well-known, much-used article with very few citations may mean
  OpenAlex has it filed under a different record

In [8]:
# Which items weren't found in OpenAlex, by type
citations[citations["citations_total"].isna()].groupby("item_type").size()

item_type
article       2
editorial     1
review       45
dtype: int64

In [9]:
# Most-cited items overall
cols = ["item_id", "year_label", "item_type", "title", "citations_total",
        "citations_first_5_years", "icite_citations"]
citations.sort_values("citations_total", ascending=False)[cols].head(20)

,item_id,year_label,item_type,title,citations_total,citations_first_5_years,icite_citations
235,JUNE-12-1-02,2013,editorial,Opinion: Science Communication to the General ...,414,52,142
295,JUNE-13-3-06,2015,article,Retention of Underrepresented Minority Faculty...,157,35,77
86,JUNE-05-2-07,2007,article,Methods for Handling Missing Data in the Behav...,131,3,79
212,JUNE-11-1-14,2012,article,Classroom Activities: Simple Strategies to Inc...,106,31,31
209,JUNE-11-1-11,2012,article,Cultivating Diversity and Competency in STEM: ...,72,13,28
431,JUNE-16-2-13,2018,article,Using ImageJ to Assess Neurite Outgrowth in Ma...,60,34,59
571,JUNE-20-2-09,2022,article,Why Students Cheat and How Understanding This ...,56,<NA>,8
29,JUNE-02-2-05,2004,article,Measuring Salivary Cortisol in the Behavioral ...,53,5,26
171,JUNE-09-2-05,2011,article,Undergraduate Neuroscience Education in the U....,50,11,41
456,JUNE-16-3-15,2018,article,A Discussion of Diversity and Inclusivity at t...,47,26,25


In [10]:
# Most-cited in their first 5 years (fairer across old and new items)
citations.sort_values("citations_first_5_years", ascending=False)[cols].head(20)

,item_id,year_label,item_type,title,citations_total,citations_first_5_years,icite_citations
235,JUNE-12-1-02,2013,editorial,Opinion: Science Communication to the General ...,414,52,142
295,JUNE-13-3-06,2015,article,Retention of Underrepresented Minority Faculty...,157,35,77
431,JUNE-16-2-13,2018,article,Using ImageJ to Assess Neurite Outgrowth in Ma...,60,34,59
212,JUNE-11-1-14,2012,article,Classroom Activities: Simple Strategies to Inc...,106,31,31
455,JUNE-16-3-14,2018,article,The New Blueprints: Undergraduate Neuroscience...,37,26,25
456,JUNE-16-3-15,2018,article,A Discussion of Diversity and Inclusivity at t...,47,26,25
519,JUNE-19-1-07,2020,article,Integrating Research into the Undergraduate Cu...,26,21,14
518,JUNE-19-1-06,2020,article,Integrating Research into the Undergraduate Cu...,22,20,15
257,JUNE-12-2-07,2014,article,Complementing Neurophysiology Education for De...,35,20,5
366,JUNE-15-1-18,2016,article,“Social” Neuroscience: Leveraging Social Media...,22,20,2


In [11]:
# How closely do OpenAlex and iCite agree? (1 = same ranking, 0 = unrelated)
both = citations.dropna(subset=["citations_total", "icite_citations"])
print(f"{len(both)} items have both counts")
both[["citations_total", "icite_citations"]].astype(float).corr(method="spearman")

656 items have both counts


,citations_total,icite_citations
citations_total,1.000000,0.935294
icite_citations,0.935294,1.000000
